# Fault Bucketing — Interactive Test & Visualization Notebook

This notebook lets you test and visualize all fault bucketing components:

1. **Data Models** — Pydantic models, FaultBucket dataclass, parsing helpers
2. **Classifier** — LLM event classifier (prompt building, fallback logic)
3. **Pipeline** — Full bucketing pipeline (with synthetic trace data)
4. **Visualization** — Fault timelines, event distribution, bucket summaries

## 0. Setup — Imports & Path Configuration

In [3]:
import sys
import json
import asyncio
import tempfile
from pathlib import Path
from datetime import datetime, timedelta, timezone

# Add the agentcert root to sys.path so that `fault_bucketing.*` and `utils.*` imports work
_NOTEBOOK_DIR = Path.cwd() if "__file__" not in dir() else Path(__file__).resolve().parent
_AGENTCERT_ROOT = _NOTEBOOK_DIR.parent.parent  # agentcert/fault_bucketing/notebooks → agentcert/
if str(_AGENTCERT_ROOT) not in sys.path:
    sys.path.insert(0, str(_AGENTCERT_ROOT))

# Core imports
from fault_bucketing.schema.data_models import (
    EventClassification,
    BatchClassificationResult,
    FaultBucket,
    safe_parse_json,
    safe_parse_python_literal,
    parse_iso_timestamp,
)
from fault_bucketing.scripts.classifier import FaultEventClassifier, _load_prompt, _load_module_config
from fault_bucketing.scripts.fault_bucketing import FaultBucketingPipeline

print("✅ All imports successful")
print(f"   Agentcert root: {_AGENTCERT_ROOT}")
print(f"   Module config:  {_load_module_config()}")

Created log directory: data
✅ All imports successful
   Agentcert root: c:\Users\machary\Downloads\Work\Projects\Infosys\Agentcert\repo\AgentCert\agentcert
   Module config:  {'classifier': {'model_name': 'extraction_model', 'temperature': 0.1, 'max_tokens': 4000, 'fallback_confidence': 0.3}, 'pipeline': {'default_batch_size': 10, 'max_filename_stem_length': 80}}


---
## 1. Data Models — Pydantic Models & Parsing Helpers

Test the core data structures used throughout the pipeline.

### 1.1 EventClassification — LLM Output Model

In [4]:
# Create an EventClassification with defaults
ec_default = EventClassification(event_id="evt-001", confidence=0.85)
print("=== Default EventClassification ===")
print(f"  event_id:       {ec_default.event_id}")
print(f"  related_faults: {ec_default.related_faults}")
print(f"  fault_detected: {ec_default.fault_detected}")
print(f"  confidence:     {ec_default.confidence}")
print()

# Create one representing a fault detection
ec_detection = EventClassification(
    event_id="evt-010",
    related_faults=["pod-delete"],
    fault_detected="pod-delete",
    detected_fault_severity="critical",
    detected_fault_target_pod="my-app-pod-xyz",
    detected_fault_namespace="default",
    detected_fault_signals=["CrashLoopBackOff", "pod not running"],
    has_quantitative_value=True,
    confidence=0.95,
)
print("=== Fault Detection Classification ===")
print(json.dumps(ec_detection.model_dump(), indent=2))
print()

# Create one representing a mitigation
ec_mitigation = EventClassification(
    event_id="evt-050",
    related_faults=["pod-delete"],
    fault_mitigated="pod-delete",
    has_qualitative_value=True,
    confidence=0.92,
)
print("=== Fault Mitigation Classification ===")
print(json.dumps(ec_mitigation.model_dump(), indent=2))

=== Default EventClassification ===
  event_id:       evt-001
  related_faults: []
  fault_detected: None
  confidence:     0.85

=== Fault Detection Classification ===
{
  "event_id": "evt-010",
  "related_faults": [
    "pod-delete"
  ],
  "fault_detected": "pod-delete",
  "detected_fault_severity": "critical",
  "detected_fault_target_pod": "my-app-pod-xyz",
  "detected_fault_namespace": "default",
  "detected_fault_signals": [
    "CrashLoopBackOff",
    "pod not running"
  ],
  "fault_mitigated": null,
  "has_quantitative_value": true,
  "has_qualitative_value": false,
  "has_cost_token_details": false,
  "confidence": 0.95
}

=== Fault Mitigation Classification ===
{
  "event_id": "evt-050",
  "related_faults": [
    "pod-delete"
  ],
  "fault_detected": null,
  "detected_fault_severity": null,
  "detected_fault_target_pod": null,
  "detected_fault_namespace": null,
  "detected_fault_signals": [],
  "fault_mitigated": "pod-delete",
  "has_quantitative_value": false,
  "has_qualit

### 1.2 BatchClassificationResult — Batch Wrapper

In [5]:
# Simulate how the LLM response is parsed into a BatchClassificationResult
batch_result = BatchClassificationResult(
    classifications=[ec_detection, ec_mitigation, ec_default]
)
print(f"Batch contains {len(batch_result.classifications)} classifications")
for c in batch_result.classifications:
    flags = []
    if c.fault_detected:
        flags.append(f"DETECTED: {c.fault_detected}")
    if c.fault_mitigated:
        flags.append(f"MITIGATED: {c.fault_mitigated}")
    if not flags:
        flags.append("event-only")
    print(f"  {c.event_id} → {', '.join(flags)} (confidence={c.confidence:.2f})")

Batch contains 3 classifications
  evt-010 → DETECTED: pod-delete (confidence=0.95)
  evt-050 → MITIGATED: pod-delete (confidence=0.92)
  evt-001 → event-only (confidence=0.85)


### 1.3 FaultBucket — Fault Lifecycle Container

In [6]:
# Create a FaultBucket with sample events
bucket = FaultBucket(
    fault_id="pod-delete",
    fault_name="pod-delete",
    severity="critical",
    target_pod="my-app-pod-xyz",
    namespace="default",
    detection_signals=["CrashLoopBackOff", "pod not running"],
    events=[
        {"id": "evt-010", "name": "k8s_pods_list", "type": "SPAN", "startTime": "2025-01-15T10:00:10Z"},
        {"id": "evt-011", "name": "investigate:pod-delete", "type": "SPAN", "startTime": "2025-01-15T10:00:20Z"},
        {"id": "evt-012", "name": "remediate:pod-delete", "type": "SPAN", "startTime": "2025-01-15T10:01:00Z"},
    ],
    status="closed",
    detected_at="2025-01-15T10:00:10Z",
    mitigated_at="2025-01-15T10:01:30Z",
    ground_truth={
        "fault_description_goal_remediation": "Pod was deleted; agent should detect and restart it."
    },
    ideal_course_of_action=["Detect missing pod", "Investigate root cause", "Restart pod"],
    ideal_tool_usage_trajectory=["k8s_pods_list", "k8s_pod_describe", "k8s_pod_restart"],
)

print("=== FaultBucket ===")
bucket_dict = bucket.to_dict()
for key, val in bucket_dict.items():
    if key == "events":
        print(f"  {key}: [{len(val)} events]")
    else:
        print(f"  {key}: {val}")

=== FaultBucket ===
  fault_id: pod-delete
  fault_name: pod-delete
  severity: critical
  target_pod: my-app-pod-xyz
  namespace: default
  detection_signals: ['CrashLoopBackOff', 'pod not running']
  status: closed
  detected_at: 2025-01-15T10:00:10Z
  mitigated_at: 2025-01-15T10:01:30Z
  ground_truth: {'fault_description_goal_remediation': 'Pod was deleted; agent should detect and restart it.'}
  ideal_course_of_action: ['Detect missing pod', 'Investigate root cause', 'Restart pod']
  ideal_tool_usage_trajectory: ['k8s_pods_list', 'k8s_pod_describe', 'k8s_pod_restart']
  event_count: 3
  events: [3 events]


---
## 2. Synthetic Trace Data

Build a realistic multi-fault trace for testing the pipeline without needing a live Azure OpenAI connection. This trace simulates two concurrent faults: `pod-delete` and `disk-fill`.

In [9]:
BASE_TIME = datetime(2025, 1, 15, 10, 0, 0, tzinfo=timezone.utc)

def ts(offset_seconds: int) -> str:
    """Generate ISO timestamp offset from BASE_TIME."""
    return (BASE_TIME + timedelta(seconds=offset_seconds)).isoformat()

# Load synthetic multi-fault trace
with open(f"{_AGENTCERT_ROOT}/data/langfuse_minio_traces/multi_fault/multi_fault_trace.json") as f:
    SYNTHETIC_TRACE = json.load(f)

print(f"✅ Synthetic trace created: {len(SYNTHETIC_TRACE)} events")
print(f"   - FAULT_DATA events: {sum(1 for e in SYNTHETIC_TRACE if e['type'] == 'FAULT_DATA')}")
print(f"   - SPAN events:       {sum(1 for e in SYNTHETIC_TRACE if e['type'] == 'SPAN')}")
print(f"   - GENERATION events:  {sum(1 for e in SYNTHETIC_TRACE if e['type'] == 'GENERATION')}")

✅ Synthetic trace created: 100 events
   - FAULT_DATA events: 3
   - SPAN events:       54
   - GENERATION events:  43


---
## 3. Classifier — LLM Prompt Construction & Fallback

Test the `FaultEventClassifier` without calling the LLM: inspect the prompt it builds and verify the fallback logic.

### 3.1 System Prompt Inspection

In [10]:
system_prompt = _load_prompt()
print(f"System prompt length: {len(system_prompt)} chars")
print("=" * 60)
print(system_prompt[:1500])
if len(system_prompt) > 1500:
    print(f"\n... ({len(system_prompt) - 1500} more chars) ...")

System prompt length: 3782 chars
You are an expert fault-event classifier for IT-Ops agent traces.

You will receive:
1. A list of currently **known faults** (may be empty initially) with their IDs, names, and target resources.
2. Optionally, a list of **injected faults** (ground truth from the chaos engineering platform) that the agent is expected to detect.
3. A **batch of trace events** (tool calls, LLM generations, agent actions) with their timestamps and content.

Your task is to:
A. **Discover fault detections**: Identify events where the agent FIRST RECOGNIZES a fault.
B. **Classify events**: Assign each event to one or more known faults.
C. **Detect mitigations**: Identify events that confirm a fault has been remediated.

## Fault Detection Rules

- Set **fault_detected** to the fault name when an event represents the agent
  FIRST RECOGNIZING that a specific fault or problem exists. This includes:
  - Explicit fault detection messages or logs from the agent
  - Agent observati

### 3.2 User Message Construction

See how the classifier builds the user message sent to the LLM for a batch of events.

In [11]:
# Instantiate classifier (no LLM call needed for prompt construction)
classifier = FaultEventClassifier(config={})

# Build known faults and injected faults for context
known_faults = {
    "pod-delete": FaultBucket(
        fault_id="pod-delete", fault_name="pod-delete",
        severity="critical", target_pod="webapp-frontend-abc123",
        namespace="production", detection_signals=["CrashLoopBackOff"],
    ),
}
injected_faults = {
    "pod-delete": FaultBucket(
        fault_id="pod-delete", fault_name="pod-delete",
        ground_truth={"fault_type": "pod-delete", "target_pod": "webapp-frontend-abc123"},
    ),
}

# Pick a small batch of events
sample_batch = [e for e in SYNTHETIC_TRACE if e["type"] != "FAULT_DATA"][:3]

user_msg = classifier.build_user_message(sample_batch, known_faults, injected_faults)
print(f"User message length: {len(user_msg)} chars")
print("=" * 60)
print(user_msg[:2000])
if len(user_msg) > 2000:
    print(f"\n... ({len(user_msg) - 2000} more chars) ...")

User message length: 5154 chars
## Known Faults

```json
[
  {
    "fault_id": "pod-delete",
    "fault_name": "pod-delete",
    "severity": "critical",
    "target_pod": "webapp-frontend-abc123",
    "namespace": "production",
    "detection_signals": [
      "CrashLoopBackOff"
    ]
  }
]
```

## Injected Faults (Ground Truth)

```json
[
  {
    "fault_id": "pod-delete",
    "fault_name": "pod-delete",
    "ground_truth": {
      "fault_type": "pod-delete",
      "target_pod": "webapp-frontend-abc123"
    }
  }
]
```

These faults were injected by the chaos engineering platform. The agent should detect and remediate them during its investigation.

## Event Batch

```json
[
  {
    "event_id": "f6ae2126-615b-4acd-8179-d453f07a22aa",
    "type": "SPAN",
    "name": "tool_call:k8s_pods_list (f6ae2126)",
    "startTime": "2026-03-17T16:51:08.948Z",
    "endTime": "2026-03-17T16:51:11.721Z",
    "parentObservationId": null,
    "input": "{\"tool_key\": \"k8s_pods_list\", \"tool_name\": \"

### 3.3 Fallback Classification

When the LLM is unavailable, the classifier assigns every event to all known faults conservatively.

In [ ]:
# Test fallback classification
multi_fault_known = {
    "pod-delete": FaultBucket(fault_id="pod-delete", fault_name="pod-delete"),
    "disk-fill": FaultBucket(fault_id="disk-fill", fault_name="disk-fill"),
}

fallback_results = classifier.fallback_classify(sample_batch, multi_fault_known)

print(f"Fallback classified {len(fallback_results)} events:")
for fc in fallback_results:
    print(f"  {fc.event_id} → related_faults={fc.related_faults}, confidence={fc.confidence}")

---
## 4. Pipeline Helpers — Unit-Level Testing

Test the pipeline's internal methods using the synthetic trace data (sorting, batching, fault detection, ground truth extraction).

### 4.1 Event Sorting

In [12]:
# Shuffle the trace and verify sorting restores chronological order
import random
shuffled = SYNTHETIC_TRACE.copy()
random.shuffle(shuffled)

sorted_events = FaultBucketingPipeline._sort_events_chronologically(shuffled)

print("Events in chronological order:")
for i, evt in enumerate(sorted_events):
    print(f"  {i+1:2d}. [{evt['type']:12s}] {evt['name']:30s}  {evt.get('startTime', 'N/A')}")

# Verify order is correct
times = [parse_iso_timestamp(e.get("startTime")) for e in sorted_events]
times_valid = [t for t in times if t is not None]
assert times_valid == sorted(times_valid), "Events are NOT sorted correctly!"
print(f"\n✅ All {len(sorted_events)} events correctly sorted by startTime")

Events in chronological order:
   1. [FAULT_DATA  ] pod_dns_error                   2026-03-17T16:51:08.000Z
   2. [FAULT_DATA  ] disk_fill                       2026-03-17T16:51:08.000Z
   3. [FAULT_DATA  ] container_kill                  2026-03-17T16:51:08.000Z
   4. [SPAN        ] tool_call:k8s_pods_list (f6ae2126)  2026-03-17T16:51:08.948Z
   5. [SPAN        ] tool_call:k8s_events_list (1a3c50e8)  2026-03-17T16:51:14.822Z
   6. [SPAN        ] tool_call:k8s_nodes_top (d22ef82d)  2026-03-17T16:51:15.890Z
   7. [GENERATION  ] cluster_scan_reasoning (f525889d)  2026-03-17T16:51:17.845Z
   8. [SPAN        ] fault_detected (f75db9bc)       2026-03-17T16:51:23.875Z
   9. [SPAN        ] fault_detected (ad45ea84)       2026-03-17T16:51:26.367Z
  10. [SPAN        ] fault_detected (375be1c8)       2026-03-17T16:51:29.416Z
  11. [GENERATION  ] triage_reasoning (30245393)     2026-03-17T16:51:35.369Z
  12. [SPAN        ] investigate:disk-fill (085ddd8f)  2026-03-17T16:51:39.434Z
  13. [SPAN   

### 4.2 FAULT_DATA Identification & Ground Truth Extraction

In [13]:
# Identify FAULT_DATA events
fault_data_events = [e for e in SYNTHETIC_TRACE if FaultBucketingPipeline._is_fault_injection_event(e)]
non_fault_events = [e for e in SYNTHETIC_TRACE if not FaultBucketingPipeline._is_fault_injection_event(e)]

print(f"FAULT_DATA events: {len(fault_data_events)}")
print(f"Non-FAULT_DATA events: {len(non_fault_events)}")
print()

# Extract ground truth from each FAULT_DATA event
for evt in fault_data_events:
    gt_bucket = FaultBucketingPipeline._extract_ground_truth(evt)
    print(f"=== Injected Fault: {gt_bucket.fault_name} ===")
    print(f"  fault_id:    {gt_bucket.fault_id}")
    print(f"  detected_at: {gt_bucket.detected_at}")
    print(f"  status:      {gt_bucket.status}")
    if gt_bucket.ground_truth:
        print(f"  ground_truth:")
        for k, v in gt_bucket.ground_truth.items():
            print(f"    {k}: {v}")
    if gt_bucket.ideal_course_of_action:
        print(f"  ideal_course_of_action:")
        for step in gt_bucket.ideal_course_of_action:
            print(f"    - {step}")
    if gt_bucket.ideal_tool_usage_trajectory:
        print(f"  ideal_tool_usage_trajectory: {gt_bucket.ideal_tool_usage_trajectory}")
    print()

FAULT_DATA events: 3
Non-FAULT_DATA events: 97

=== Injected Fault: disk_fill ===
  fault_id:    disk_fill
  detected_at: 2026-03-17T16:51:08.000Z
  status:      active
  ground_truth:
    fault_description_goal_remediation: {'symptoms': [], 'goals': '', 'remediation': ''}

=== Injected Fault: container_kill ===
  fault_id:    container_kill
  detected_at: 2026-03-17T16:51:08.000Z
  status:      active
  ground_truth:
    fault_description_goal_remediation: {'symptoms': [], 'goals': '', 'remediation': ''}

=== Injected Fault: pod_dns_error ===
  fault_id:    pod_dns_error
  detected_at: 2026-03-17T16:51:08.000Z
  status:      active
  ground_truth:
    fault_description_goal_remediation: {'symptoms': [], 'goals': '', 'remediation': ''}



### 4.3 Event Batching

In [14]:
# Test batching with different sizes
for batch_size in [5, 10, 16]:
    batches = FaultBucketingPipeline._create_event_batches(non_fault_events, batch_size)
    batch_sizes = [len(b) for b in batches]
    print(f"batch_size={batch_size:2d} → {len(batches)} batches, sizes={batch_sizes}")

# Verify all events are preserved
batches = FaultBucketingPipeline._create_event_batches(non_fault_events, 5)
total = sum(len(b) for b in batches)
assert total == len(non_fault_events), "Events lost during batching!"
print(f"\n✅ All {total} events preserved across batches")

batch_size= 5 → 20 batches, sizes=[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 2]
batch_size=10 → 10 batches, sizes=[10, 10, 10, 10, 10, 10, 10, 10, 10, 7]
batch_size=16 → 7 batches, sizes=[16, 16, 16, 16, 16, 16, 1]

✅ All 97 events preserved across batches


### 4.4 Bucket Management — Event Placement & Fault Closure

In [15]:
# Write trace to a temp file and create a pipeline instance to test helpers
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False, dir=tempfile.gettempdir()) as f:
    json.dump(SYNTHETIC_TRACE, f)
    trace_path = f.name

output_dir = tempfile.mkdtemp(prefix="fault_bucketing_test_")
pipeline = FaultBucketingPipeline(
    trace_file_path=trace_path,
    output_dir=output_dir,
    config={},
    batch_size=5,
)

# Set up active faults manually
pipeline.active_faults["pod-delete"] = FaultBucket(
    fault_id="pod-delete", fault_name="pod-delete", severity="critical",
    events=[], status="active", detected_at=ts(25),
)
pipeline.active_faults["disk-fill"] = FaultBucket(
    fault_id="disk-fill", fault_name="disk-fill", severity="high",
    events=[], status="active", detected_at=ts(42),
)

# Test event placement
test_event = {"id": "evt-test", "name": "investigate:pod-delete"}
classification = EventClassification(
    event_id="evt-test",
    related_faults=["pod-delete"],
    confidence=0.9,
)
pipeline._place_event_in_buckets(test_event, classification)
print(f"pod-delete bucket events: {len(pipeline.active_faults['pod-delete'].events)}")
print(f"disk-fill bucket events:  {len(pipeline.active_faults['disk-fill'].events)}")

# Test an event that doesn't match any bucket
orphan_event = {"id": "evt-orphan", "name": "unrelated"}
orphan_class = EventClassification(
    event_id="evt-orphan",
    related_faults=["nonexistent-fault"],
    confidence=0.5,
)
pipeline._place_event_in_buckets(orphan_event, orphan_class)
print(f"Unclassified events:      {len(pipeline.unclassified_events)}")

# Test fault closure
print(f"\nBefore closure:")
print(f"  Active faults: {list(pipeline.active_faults.keys())}")
print(f"  Closed faults: {list(pipeline.closed_faults.keys())}")

pipeline._close_fault("pod-delete", mitigated_at=ts(100))

print(f"\nAfter closing pod-delete:")
print(f"  Active faults: {list(pipeline.active_faults.keys())}")
print(f"  Closed faults: {list(pipeline.closed_faults.keys())}")
print(f"  pod-delete status: {pipeline.closed_faults['pod-delete'].status}")
print(f"  pod-delete mitigated_at: {pipeline.closed_faults['pod-delete'].mitigated_at}")

2026-03-23 14:05:30,564 - [fault_bucketing.py : _close_fault : 277] - INFO - Fault bucket closed: pod-delete (1 events)


pod-delete bucket events: 1
disk-fill bucket events:  0
Unclassified events:      1

Before closure:
  Active faults: ['pod-delete', 'disk-fill']
  Closed faults: []

After closing pod-delete:
  Active faults: ['disk-fill']
  Closed faults: ['pod-delete']
  pod-delete status: closed
  pod-delete mitigated_at: 2025-01-15T10:01:40+00:00


### 4.5 Ground Truth Enrichment

In [16]:
# Set up injected faults for enrichment
pipeline.injected_faults = {}
for evt in fault_data_events:
    gt = FaultBucketingPipeline._extract_ground_truth(evt)
    pipeline.injected_faults[gt.fault_id] = gt

# Create a detected bucket without ground truth
detected_bucket = FaultBucket(
    fault_id="pod-delete", fault_name="pod-delete",
    severity="critical", events=[], status="active",
)
print(f"Before enrichment:")
print(f"  ground_truth: {detected_bucket.ground_truth}")
print(f"  ideal_course_of_action: {detected_bucket.ideal_course_of_action}")

# Enrich with ground truth
pipeline._enrich_bucket_with_ground_truth(detected_bucket)

print(f"\nAfter enrichment:")
print(f"  ground_truth: {json.dumps(detected_bucket.ground_truth, indent=4)}")
print(f"  ideal_course_of_action: {detected_bucket.ideal_course_of_action}")
print(f"  ideal_tool_usage_trajectory: {detected_bucket.ideal_tool_usage_trajectory}")

Before enrichment:
  ground_truth: None
  ideal_course_of_action: None

After enrichment:
  ground_truth: null
  ideal_course_of_action: None
  ideal_tool_usage_trajectory: None


---
## 5. Simulated Full Pipeline Run

Run the complete pipeline end-to-end using simulated LLM classifications (no Azure OpenAI needed). We mock the classifier to produce realistic classifications for each event.

In [17]:
from unittest.mock import AsyncMock, patch

# Define mock classifications that simulate realistic LLM output
MOCK_CLASSIFICATIONS = {
    "evt-001": EventClassification(event_id="evt-001", related_faults=[], confidence=0.7),
    "evt-002": EventClassification(event_id="evt-002", related_faults=[], confidence=0.6),
    "evt-003": EventClassification(event_id="evt-003", related_faults=[], confidence=0.8),
    "evt-004": EventClassification(
        event_id="evt-004", related_faults=["pod-delete"],
        fault_detected="pod-delete", detected_fault_severity="critical",
        detected_fault_target_pod="webapp-frontend-abc123",
        detected_fault_namespace="production",
        detected_fault_signals=["CrashLoopBackOff", "pod not running"],
        has_qualitative_value=True, confidence=0.95,
    ),
    "evt-005": EventClassification(
        event_id="evt-005", related_faults=["pod-delete"],
        has_quantitative_value=True, confidence=0.9,
    ),
    "evt-006": EventClassification(
        event_id="evt-006", related_faults=[],
        has_quantitative_value=True, confidence=0.85,
    ),
    "evt-007": EventClassification(
        event_id="evt-007", related_faults=["disk-fill"],
        fault_detected="disk-fill", detected_fault_severity="high",
        detected_fault_target_pod="data-processor-xyz789",
        detected_fault_namespace="production",
        detected_fault_signals=["disk usage >95%", "write errors"],
        has_quantitative_value=True, has_qualitative_value=True, confidence=0.93,
    ),
    "evt-008": EventClassification(
        event_id="evt-008", related_faults=["pod-delete"], confidence=0.88,
    ),
    "evt-009": EventClassification(
        event_id="evt-009", related_faults=["pod-delete"],
        has_qualitative_value=True, confidence=0.92,
    ),
    "evt-010": EventClassification(
        event_id="evt-010", related_faults=["disk-fill"],
        has_quantitative_value=True, confidence=0.87,
    ),
    "evt-011": EventClassification(
        event_id="evt-011", related_faults=["disk-fill"],
        has_qualitative_value=True, confidence=0.90,
    ),
    "evt-012": EventClassification(
        event_id="evt-012", related_faults=["pod-delete"], confidence=0.94,
    ),
    "evt-013": EventClassification(
        event_id="evt-013", related_faults=["pod-delete"],
        has_quantitative_value=True, confidence=0.88,
    ),
    "evt-014": EventClassification(
        event_id="evt-014", related_faults=["pod-delete"],
        fault_mitigated="pod-delete",
        has_qualitative_value=True, confidence=0.96,
    ),
    "evt-015": EventClassification(
        event_id="evt-015", related_faults=["disk-fill"], confidence=0.91,
    ),
    "evt-016": EventClassification(
        event_id="evt-016", related_faults=["disk-fill"],
        has_quantitative_value=True, confidence=0.89,
    ),
    "evt-017": EventClassification(
        event_id="evt-017", related_faults=["disk-fill"],
        fault_mitigated="disk-fill",
        has_qualitative_value=True, confidence=0.95,
    ),
    "evt-018": EventClassification(
        event_id="evt-018", related_faults=["pod-delete", "disk-fill"],
        has_cost_token_details=True, has_qualitative_value=True, confidence=0.85,
    ),
}

async def mock_classify_batch(batch, known_faults, injected_faults):
    """Return pre-defined classifications for each event in the batch."""
    return [
        MOCK_CLASSIFICATIONS.get(
            evt.get("id", ""),
            EventClassification(event_id=evt.get("id", "unknown"), confidence=0.5),
        )
        for evt in batch
    ]

# Run the full pipeline with mocked classifier
sim_pipeline = FaultBucketingPipeline(
    trace_file_path=trace_path,
    output_dir=output_dir,
    config={},
    batch_size=5,
)

# Patch the classifier's classify_batch method
sim_pipeline._classifier.classify_batch = mock_classify_batch

# Run (use nest_asyncio for notebook compatibility)
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

buckets = asyncio.get_event_loop().run_until_complete(sim_pipeline.run())

print(f"\n{'='*60}")
print(f"PIPELINE RESULTS")
print(f"{'='*60}")
print(f"  Injected faults:    {len(sim_pipeline.injected_faults)}")
print(f"  Total buckets:      {len(buckets)}")
print(f"  Active faults:      {list(sim_pipeline.active_faults.keys())}")
print(f"  Closed faults:      {list(sim_pipeline.closed_faults.keys())}")
print(f"  Unclassified:       {len(sim_pipeline.unclassified_events)}")
print()

for fid, b in buckets.items():
    print(f"  [{b.status.upper():6s}] {fid}")
    print(f"    events:      {len(b.events)}")
    print(f"    severity:    {b.severity}")
    print(f"    target_pod:  {b.target_pod}")
    print(f"    detected_at: {b.detected_at}")
    print(f"    mitigated_at:{b.mitigated_at}")
    print(f"    ground_truth:{b.ground_truth is not None}")
    print()

2026-03-23 14:06:21,738 - [fault_bucketing.py : _load_trace : 135] - INFO - Loaded 100 events from tmp5n_7p9j7.json
2026-03-23 14:06:21,748 - [fault_bucketing.py : run : 314] - INFO - Fault injected (FAULT_DATA): disk_fill (ground_truth=True)
2026-03-23 14:06:21,749 - [fault_bucketing.py : run : 314] - INFO - Fault injected (FAULT_DATA): container_kill (ground_truth=True)
2026-03-23 14:06:21,750 - [fault_bucketing.py : run : 314] - INFO - Fault injected (FAULT_DATA): pod_dns_error (ground_truth=True)
2026-03-23 14:06:21,752 - [fault_bucketing.py : run : 330] - INFO - Processing 97 events via LLM (batch_size=5)
2026-03-23 14:06:21,761 - [fault_bucketing.py : run : 415] - INFO - Batch 1/20 processed (5 events)
2026-03-23 14:06:21,761 - [fault_bucketing.py : run : 415] - INFO - Batch 2/20 processed (5 events)
2026-03-23 14:06:21,762 - [fault_bucketing.py : run : 415] - INFO - Batch 3/20 processed (5 events)
2026-03-23 14:06:21,763 - [fault_bucketing.py : run : 415] - INFO - Batch 4/20 pro


PIPELINE RESULTS
  Injected faults:    3
  Total buckets:      1
  Active faults:      []
  Closed faults:      ['single_fault']
  Unclassified:       97

  [CLOSED] single_fault
    events:      97
    severity:    None
    target_pod:  None
    detected_at: 2026-03-17T16:51:08.948Z
    mitigated_at:2026-03-17T17:21:46.843Z
    ground_truth:False



---
## 6. Visualization

Visualize the fault bucketing results: event timelines, distribution across buckets, and pipeline summary.

### 6.1 Event Distribution per Fault Bucket

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("⚠️  matplotlib not available — install with: pip install matplotlib")
    print("   Falling back to text-based visualization.\n")

if HAS_MATPLOTLIB:
    # Bar chart: events per bucket
    fault_names = list(buckets.keys())
    event_counts = [len(b.events) for b in buckets.values()]
    colors = ["#e74c3c" if b.severity == "critical" else "#f39c12" for b in buckets.values()]

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(fault_names, event_counts, color=colors, edgecolor="white", height=0.5)
    ax.set_xlabel("Number of Events")
    ax.set_title("Events per Fault Bucket")

    for bar, count in zip(bars, event_counts):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                str(count), va="center", fontweight="bold")

    # Add severity legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="#e74c3c", label="Critical"),
        Patch(facecolor="#f39c12", label="High"),
    ]
    ax.legend(handles=legend_elements, title="Severity", loc="lower right")
    plt.tight_layout()
    plt.show()
else:
    print("Events per Fault Bucket:")
    for fid, b in buckets.items():
        bar = "█" * len(b.events)
        print(f"  {fid:20s} | {bar} ({len(b.events)} events, {b.severity})")

### 6.2 Fault Lifecycle Timeline

Visualize when each fault was detected and mitigated, with events plotted along the timeline.

In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(12, 4))

    fault_colors = {"pod-delete": "#e74c3c", "disk-fill": "#f39c12"}
    y_positions = {fid: i for i, fid in enumerate(buckets.keys())}

    for fid, b in buckets.items():
        y = y_positions[fid]
        color = fault_colors.get(fid, "#3498db")

        # Draw lifecycle bar (detection → mitigation)
        det_time = parse_iso_timestamp(b.detected_at)
        mit_time = parse_iso_timestamp(b.mitigated_at)
        if det_time and mit_time:
            duration = (mit_time - det_time).total_seconds()
            ax.barh(y, duration, left=mdates.date2num(det_time),
                    color=color, alpha=0.3, height=0.4, label=f"{fid} lifecycle")

        # Plot individual events
        event_times = []
        event_types = []
        for evt in b.events:
            t = parse_iso_timestamp(evt.get("startTime"))
            if t:
                event_times.append(t)
                event_types.append(evt.get("type", "SPAN"))

        if event_times:
            markers = {"SPAN": "o", "GENERATION": "s", "FAULT_DATA": "D"}
            for et, etype in zip(event_times, event_types):
                marker = markers.get(etype, "o")
                ax.plot(mdates.date2num(et), y, marker=marker, color=color,
                        markersize=8, markeredgecolor="white", markeredgewidth=1)

        # Annotate detection and mitigation points
        if det_time:
            ax.annotate("DETECTED", (mdates.date2num(det_time), y),
                        textcoords="offset points", xytext=(5, 15),
                        fontsize=7, color=color, fontweight="bold")
        if mit_time:
            ax.annotate("MITIGATED", (mdates.date2num(mit_time), y),
                        textcoords="offset points", xytext=(5, 15),
                        fontsize=7, color="green", fontweight="bold")

    ax.set_yticks(list(y_positions.values()))
    ax.set_yticklabels(list(y_positions.keys()))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax.set_xlabel("Time")
    ax.set_title("Fault Lifecycle Timeline")
    ax.grid(axis="x", alpha=0.3)

    # Legend for event types
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker="o", color="gray", label="SPAN", linestyle="None", markersize=8),
        Line2D([0], [0], marker="s", color="gray", label="GENERATION", linestyle="None", markersize=8),
        Line2D([0], [0], marker="D", color="gray", label="FAULT_DATA", linestyle="None", markersize=8),
    ]
    ax.legend(handles=legend_elements, loc="upper right", title="Event Type")
    plt.tight_layout()
    plt.show()
else:
    print("Fault Lifecycle Timeline (text):")
    for fid, b in buckets.items():
        det = b.detected_at or "?"
        mit = b.mitigated_at or "ongoing"
        print(f"  {fid}: {det} → {mit}  ({len(b.events)} events, {b.status})")

### 6.3 Event Type Breakdown per Bucket

In [ ]:
from collections import Counter

if HAS_MATPLOTLIB:
    fig, axes = plt.subplots(1, len(buckets), figsize=(5 * len(buckets), 4))
    if len(buckets) == 1:
        axes = [axes]

    for ax, (fid, b) in zip(axes, buckets.items()):
        type_counts = Counter(evt.get("type", "UNKNOWN") for evt in b.events)
        labels = list(type_counts.keys())
        sizes = list(type_counts.values())
        colors_pie = {"SPAN": "#3498db", "GENERATION": "#2ecc71", "FAULT_DATA": "#9b59b6"}
        pie_colors = [colors_pie.get(l, "#95a5a6") for l in labels]

        wedges, texts, autotexts = ax.pie(
            sizes, labels=labels, colors=pie_colors,
            autopct="%1.0f%%", startangle=90, textprops={"fontsize": 9},
        )
        ax.set_title(f"{fid}\n({len(b.events)} events)")

    plt.suptitle("Event Type Breakdown per Fault Bucket", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Event Type Breakdown:")
    for fid, b in buckets.items():
        type_counts = Counter(evt.get("type", "UNKNOWN") for evt in b.events)
        print(f"  {fid}: {dict(type_counts)}")

### 6.4 Classification Confidence Distribution

In [ ]:
# Analyze the confidence of our mock classifications
all_confidences = [c.confidence for c in MOCK_CLASSIFICATIONS.values()]
detection_confidences = [c.confidence for c in MOCK_CLASSIFICATIONS.values() if c.fault_detected]
mitigation_confidences = [c.confidence for c in MOCK_CLASSIFICATIONS.values() if c.fault_mitigated]
regular_confidences = [c.confidence for c in MOCK_CLASSIFICATIONS.values()
                       if not c.fault_detected and not c.fault_mitigated]

if HAS_MATPLOTLIB:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram
    ax1.hist(all_confidences, bins=10, range=(0, 1), color="#3498db",
             edgecolor="white", alpha=0.8)
    ax1.axvline(sum(all_confidences)/len(all_confidences), color="#e74c3c",
                linestyle="--", label=f"Mean: {sum(all_confidences)/len(all_confidences):.2f}")
    ax1.set_xlabel("Confidence")
    ax1.set_ylabel("Count")
    ax1.set_title("Classification Confidence Distribution")
    ax1.legend()

    # Box plot by category
    box_data = []
    box_labels = []
    if detection_confidences:
        box_data.append(detection_confidences)
        box_labels.append(f"Detection\n(n={len(detection_confidences)})")
    if mitigation_confidences:
        box_data.append(mitigation_confidences)
        box_labels.append(f"Mitigation\n(n={len(mitigation_confidences)})")
    if regular_confidences:
        box_data.append(regular_confidences)
        box_labels.append(f"Regular\n(n={len(regular_confidences)})")

    bp = ax2.boxplot(box_data, labels=box_labels, patch_artist=True)
    colors_box = ["#e74c3c", "#27ae60", "#3498db"]
    for patch, color in zip(bp["boxes"], colors_box[:len(box_data)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax2.set_ylabel("Confidence")
    ax2.set_title("Confidence by Classification Type")

    plt.tight_layout()
    plt.show()
else:
    print("Classification Confidence Summary:")
    print(f"  All:        mean={sum(all_confidences)/len(all_confidences):.3f}, "
          f"min={min(all_confidences):.3f}, max={max(all_confidences):.3f}")
    if detection_confidences:
        print(f"  Detection:  mean={sum(detection_confidences)/len(detection_confidences):.3f}")
    if mitigation_confidences:
        print(f"  Mitigation: mean={sum(mitigation_confidences)/len(mitigation_confidences):.3f}")

---
## 7. Output Inspection

Examine the output files generated by the pipeline (per-fault bucket JSONs and manifest).

In [ ]:
# List output files
output_path = Path(output_dir)
print(f"Output directory: {output_path}")
print(f"{'='*60}")

output_files = sorted(output_path.glob("*.json"))
for f in output_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:50s} ({size_kb:.1f} KB)")

# Load and display the manifest
manifest_files = [f for f in output_files if "manifest" in f.name]
if manifest_files:
    print(f"\n{'='*60}")
    print("MANIFEST CONTENTS:")
    print(f"{'='*60}")
    with open(manifest_files[0], "r") as f:
        manifest = json.load(f)
    # Print without full event data
    manifest_display = {k: v for k, v in manifest.items()}
    print(json.dumps(manifest_display, indent=2, default=str))

### 7.1 Inspect Individual Bucket Files

In [ ]:
# Load and display each bucket file (metadata only, events summarized)
bucket_files = [f for f in output_files if "bucket" in f.name]

for bf in bucket_files:
    with open(bf, "r") as f:
        data = json.load(f)

    print(f"\n{'='*60}")
    print(f"FILE: {bf.name}")
    print(f"{'='*60}")
    # Show metadata
    for key in ["fault_id", "fault_name", "severity", "target_pod", "namespace",
                 "status", "detected_at", "mitigated_at", "event_count"]:
        print(f"  {key:25s}: {data.get(key)}")

    # Show ground truth if present
    if data.get("ground_truth"):
        print(f"  {'ground_truth':25s}: ✅ present")
    if data.get("ideal_course_of_action"):
        print(f"  {'ideal_course_of_action':25s}: {data['ideal_course_of_action']}")

    # Show event names in order
    print(f"\n  Event sequence:")
    for i, evt in enumerate(data.get("events", [])):
        print(f"    {i+1:2d}. [{evt.get('type', '?'):12s}] {evt.get('name', '?')}")

---
## 8. Ground Truth Comparison

Compare the pipeline's detected faults with the injected ground truth to validate correctness.

In [ ]:
print("GROUND TRUTH vs DETECTED FAULTS")
print("=" * 60)

injected_names = set(sim_pipeline.injected_faults.keys())
detected_names = set(buckets.keys())

print(f"\n  Injected faults:  {sorted(injected_names)}")
print(f"  Detected faults:  {sorted(detected_names)}")

matched = injected_names & detected_names
missed = injected_names - detected_names
extra = detected_names - injected_names

print(f"\n  ✅ Matched:  {sorted(matched) if matched else 'None'}")
print(f"  ❌ Missed:   {sorted(missed) if missed else 'None'}")
print(f"  ➕ Extra:    {sorted(extra) if extra else 'None'}")

# Detailed per-fault comparison
print(f"\n{'='*60}")
print("DETAILED COMPARISON")
print(f"{'='*60}")

for fid in sorted(injected_names | detected_names):
    print(f"\n--- {fid} ---")
    injected = sim_pipeline.injected_faults.get(fid)
    detected = buckets.get(fid)

    if injected and detected:
        # Check ground truth enrichment
        gt_enriched = detected.ground_truth is not None
        print(f"  Status:            {detected.status}")
        print(f"  Ground truth:      {'✅ enriched' if gt_enriched else '❌ missing'}")
        print(f"  Events:            {len(detected.events)}")

        # Check ideal tool trajectory match
        if detected.ideal_tool_usage_trajectory:
            actual_tools = [e.get("name", "") for e in detected.events if e.get("type") == "SPAN"]
            ideal_tools = detected.ideal_tool_usage_trajectory
            tool_match = sum(1 for t in ideal_tools if any(t in a for a in actual_tools))
            print(f"  Tool trajectory:   {tool_match}/{len(ideal_tools)} ideal tools used")
            print(f"    Ideal:  {ideal_tools}")
            print(f"    Actual: {actual_tools}")
    elif injected:
        print(f"  ❌ Injected but NOT detected")
    else:
        print(f"  ➕ Detected but NOT in ground truth")

---
## 9. Run with Live LLM (Optional)

If you have Azure OpenAI credentials configured, uncomment and run the cell below to execute the pipeline against the real LLM instead of mocked classifications.

Set these environment variables before running:
- `AZURE_OPENAI_ENDPOINT`
- `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_API_VERSION`
- `AZURE_OPENAI_CHAT_DEPLOYMENT_NAME`

In [ ]:
# === UNCOMMENT BELOW TO RUN WITH LIVE LLM ===

# import os
# # Verify env vars are set
# required_vars = [
#     "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY",
#     "AZURE_OPENAI_API_VERSION", "AZURE_OPENAI_CHAT_DEPLOYMENT_NAME",
# ]
# missing = [v for v in required_vars if not os.environ.get(v)]
# if missing:
#     print(f"❌ Missing environment variables: {missing}")
# else:
#     print("✅ All environment variables set")
#
#     live_output_dir = tempfile.mkdtemp(prefix="fault_bucketing_live_")
#     live_pipeline = FaultBucketingPipeline(
#         trace_file_path=trace_path,
#         output_dir=live_output_dir,
#         batch_size=5,
#     )
#     live_buckets = asyncio.get_event_loop().run_until_complete(live_pipeline.run())
#
#     print(f"\nLive Pipeline Results:")
#     print(f"  Buckets: {len(live_buckets)}")
#     print(f"  Input tokens:  {live_pipeline.total_input_tokens}")
#     print(f"  Output tokens: {live_pipeline.total_output_tokens}")
#     for fid, b in live_buckets.items():
#         print(f"  [{b.status}] {fid}: {len(b.events)} events")

print("ℹ️  Live LLM section is commented out. Uncomment to run with Azure OpenAI.")

---
## 10. Cleanup

In [ ]:
import shutil

# Clean up temp files
Path(trace_path).unlink(missing_ok=True)
shutil.rmtree(output_dir, ignore_errors=True)

print("✅ Temporary files cleaned up")